# Saddle Point Navigation — 6-Robot Pentagon Formation

**5 robots on a ring + 1 at the center.** Uses exact quadratic least-squares fitting (6 equations, 6 unknowns) to directly estimate the gradient and Hessian of a scalar field. A Newton step then navigates toward the saddle point. No formation rotation control.

**Formation:** Pentagon (radius D) with a central robot.

**Field estimation:** Direct 6×6 least-squares solve. The basis is:
$$\mathbf{\varphi}(\bar{\mathbf{r}}) = \left[1, x, y, \tfrac{x^2}{2}, xy, \tfrac{y^2}{2}\right]^T$$

This gives coefficients that are exactly the Taylor expansion data at the center: $\theta = [\sigma(c), \partial_x, \partial_y, H_{11}, H_{12}, H_{22}]^T$.

## Imports

In [106]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

## Scalar Field: Log-Sum-Exp Saddle

In [107]:
class ScalarField:
    """Scalar field: soft-max of two Gaussians. Saddle at origin."""
    
    def __init__(self):
        # Two Gaussians centered at (-2, 0) and (2, 0)
        self.g1_center = np.array([-2.0, 0.0])
        self.g2_center = np.array([2.0, 0.0])
    
    def evaluate(self, x, y):
        """Log-sum-exp soft max of two Gaussians."""
        g1 = -((x + 2)**2 + y**2) / 2
        g2 = -((x - 2)**2 + y**2) / 2
        mx = np.maximum(g1, g2)
        return mx + np.log(np.exp(g1 - mx) + np.exp(g2 - mx))
    
    def plot_contour(self, ax, x_range=(-3, 3), y_range=(-3, 3), levels=20):
        """Draw 2D contour on given axis."""
        x = np.linspace(x_range[0], x_range[1], 200)
        y = np.linspace(y_range[0], y_range[1], 200)
        X, Y = np.meshgrid(x, y)
        Z = self.evaluate(X, Y)
        cs = ax.contourf(X, Y, Z, levels=levels, cmap='viridis', alpha=0.6)
        ax.contour(X, Y, Z, levels=levels, colors='gray', linewidths=0.5, alpha=0.3)
        return cs
    
    def plot_surface(self, ax, x_range=(-3, 3), y_range=(-3, 3), grid_res=100):
        """Draw 3D surface on given axis."""
        x = np.linspace(x_range[0], x_range[1], grid_res)
        y = np.linspace(y_range[0], y_range[1], grid_res)
        X, Y = np.meshgrid(x, y)
        Z = self.evaluate(X, Y)
        surf = ax.plot_surface(X, Y, Z, cmap='viridis', alpha=0.7, edgecolor='none')
        return surf

## Pentagon + Center Formation

**5 robots uniformly spaced on a circle** of radius D at angles $\varphi_0 + \frac{2\pi k}{5}$ for $k=0,1,2,3,4$.

**1 robot at the center** of the formation.

Total: **6 robots**. The formation has a fixed base angle (no rotation control).

In [108]:
class PentagonFormation:
    """5 robots on a pentagon ring + 1 at center."""
    
    def __init__(self, radius=0.25, base_angle=0.0):
        """
        Args:
            radius: distance from centroid to ring robots
            base_angle: fixed orientation of the pentagon (no rotation control)
        """
        self.radius = radius
        self.base_angle = base_angle
    
    def robot_positions(self, centroid):
        """Return positions of all 6 robots centered at centroid.
        
        Returns:
            np.ndarray of shape (6, 2): rows are [x, y] coordinates.
            Rows 0-4 are ring robots, row 5 is the center.
        """
        centroid = np.asarray(centroid)
        # 5 ring robots
        angles = self.base_angle + 2 * np.pi * np.arange(5) / 5
        ring_positions = centroid[np.newaxis, :] + self.radius * np.column_stack(
            [np.cos(angles), np.sin(angles)]
        )
        # 1 center robot
        center = centroid[np.newaxis, :]
        return np.vstack([ring_positions, center])
    
    def relative_positions(self, centroid):
        """Return relative positions r̄ᵢ = robotᵢ - centroid.
        
        Returns:
            np.ndarray of shape (6, 2).
        """
        robots = self.robot_positions(centroid)
        return robots - np.asarray(centroid)[np.newaxis, :]

## Quadratic Estimator: Least-Squares Fit

**Basis:** $\mathbf{\varphi}(\bar{\mathbf{r}}) = [1, x, y, \frac{x^2}{2}, xy, \frac{y^2}{2}]^T$

**System:** With 6 robots and 6 basis functions, we have a 6×6 matrix $\Phi$ and can solve exactly:
$$\theta = \Phi^{-1} \mathbf{s}$$

**Coefficient meaning:** $\theta = [\sigma(c), \partial_x, \partial_y, H_{11}, H_{12}, H_{22}]^T$

where $H$ is the Hessian of $\sigma$ at the formation center.

In [109]:
class QuadraticEstimator:
    """Fit a quadratic model f(x,y) using 6 robot readings.
    
    The quadratic model yields the Hessian directly from the coefficients.
    """
    
    @staticmethod
    def basis(rx, ry):
        """Evaluate basis φ(r̄) at relative position (rx, ry).
        
        Returns:
            np.ndarray of shape (6,): [1, rx, ry, rx²/2, rx*ry, ry²/2]
        """
        return np.array([1, rx, ry, rx**2 / 2, rx * ry, ry**2 / 2], dtype=float)
    
    def fit(self, relative_positions, readings):
        """Fit the 6×6 least-squares system.
        
        Args:
            relative_positions: np.ndarray of shape (6, 2), rows are (rx, ry) values
            readings: np.ndarray of shape (6,), scalar field values s_i
        
        Returns:
            theta: np.ndarray of shape (6,), coefficients [σ(c), ∂x, ∂y, H11, H12, H22]
        """
        # Build 6×6 Vandermonde matrix
        Phi = np.array([
            self.basis(r[0], r[1])
            for r in relative_positions
        ])
        # Solve exactly (6 equations, 6 unknowns)
        try:
            theta = np.linalg.solve(Phi, readings)
        except np.linalg.LinAlgError:
            # Fallback to least-squares if singular (should not happen for pentagon)
            theta = np.linalg.lstsq(Phi, readings, rcond=None)[0]
        return theta
    
    def extract_gradient(self, theta):
        """Extract gradient [∂σ/∂x, ∂σ/∂y] from coefficient vector.
        
        Args:
            theta: shape (6,)
        
        Returns:
            np.ndarray of shape (2,): [∂σ/∂x, ∂σ/∂y]
        """
        return theta[1:3]
    
    def extract_hessian(self, theta):
        """Extract Hessian from coefficient vector.
        
        Args:
            theta: shape (6,)
        
        Returns:
            H: np.ndarray of shape (2, 2), symmetric Hessian
        """
        H11 = theta[3]
        H12 = theta[4]
        H22 = theta[5]
        return np.array([[H11, H12], [H12, H22]])

## Newton Stepper

**Compute a Newton step:** $\Delta = -H^{-1} \nabla\sigma$

At a saddle point, the Hessian $H$ is indefinite (one positive and one negative eigenvalue). The step points toward the saddle regardless of the direction of the gradient.

**Step clipping:** Magnitude is clipped to a maximum unit length to keep iterations numerically stable.

In [110]:
class NewtonStepper:
    """Compute Newton steps toward saddle points."""
    
    def __init__(self, step_size=0.02, max_step=1.0):
        """
        Args:
            step_size: scalar multiplier on the Newton step
            max_step: maximum magnitude of Newton delta before clipping
        """
        self.step_size = step_size
        self.max_step = max_step
    
    def compute_step(self, gradient, hessian):
        """Compute a single Newton step.
        
        Args:
            gradient: np.ndarray of shape (2,), ∇σ at the centroid
            hessian: np.ndarray of shape (2, 2), H at the centroid
        
        Returns:
            step: np.ndarray of shape (2,), displacement to add to centroid
        """
        # Try to solve H * delta = -grad
        det = np.linalg.det(hessian)
        if np.abs(det) > 1e-10:
            delta = -np.linalg.solve(hessian, gradient)
        else:
            # Singular: use pseudo-inverse
            delta = -np.linalg.pinv(hessian) @ gradient
        
        # Clip magnitude
        delta_norm = np.linalg.norm(delta)
        if delta_norm > self.max_step:
            delta = delta / delta_norm
        
        return self.step_size * delta

In [111]:
class AnisotropicNewtonStepper:
    """Drop-in replacement for NewtonStepper.

    Splits the Newton step in the Hessian eigenbasis and damps each
    eigendirection separately:
        across-trench (positive curvature) -> gain_perp  (large = fast snap)
        along-trench  (negative curvature) -> gain_along (small = slow climb)
    gain_perp == gain_along recovers the plain damped Newton step exactly.
    """
    def __init__(self, gain_perp=0.5, gain_along=0.02, max_step=1.0):
        self.gain_perp = gain_perp
        self.gain_along = gain_along
        self.max_step = max_step

    def compute_step(self, gradient, hessian):
        lam, V = np.linalg.eigh(hessian)   # symmetric -> orthonormal V, ascending lam
        eps = 1e-9
        if lam[0] * lam[1] >= -eps or np.min(np.abs(lam)) < eps:
            # not a clean local saddle: fall back to damped Newton
            step = self.gain_along * (-np.linalg.pinv(hessian) @ gradient)
        else:
            step = np.zeros(2)
            for i in (0, 1):
                vi = V[:, i]
                gain = self.gain_perp if lam[i] > 0 else self.gain_along
                step = step + gain * (-(vi @ gradient) / lam[i]) * vi
        n = np.linalg.norm(step)
        if n > self.max_step:
            step = step / n * self.max_step
        return step

## Saddle Point Navigator

**Main simulation loop:**
1. Place 6 robots (pentagon + center) around current centroid
2. Read scalar field at each robot
3. Fit quadratic model → extract gradient and Hessian
4. Compute Newton step
5. Update centroid
6. Record history for diagnostics

**No rotation control:** The formation orientation is fixed throughout.

In [112]:
class SaddlePointNavigator6:
    """Navigate a 6-robot formation toward a saddle point using Newton's method."""
    
    def __init__(self, formation, estimator, stepper, field, n_iterations=200):
        """
        Args:
            formation: PentagonFormation instance
            estimator: QuadraticEstimator instance
            stepper: NewtonStepper instance
            field: ScalarField instance
            n_iterations: number of Newton steps to take
        """
        self.formation = formation
        self.estimator = estimator
        self.stepper = stepper
        self.field = field
        self.n_iterations = n_iterations
        self.history = {
            'centroids': [],
            'gradients': [],
            'hessians': [],
        }
    
    def navigate(self, start_centroid):
        """Run the navigation loop.
        
        Args:
            start_centroid: initial position (x, y)
        
        Returns:
            final_centroid: position after n_iterations steps
        """
        centroid = np.array(start_centroid, dtype=float)
        self.initial_centroid = centroid.copy()
        
        for iteration in range(self.n_iterations):
            # Get robot positions and relative positions
            robots = self.formation.robot_positions(centroid)
            rel_pos = self.formation.relative_positions(centroid)
            
            # Read scalar field at each robot
            readings = np.array([
                self.field.evaluate(r[0], r[1])
                for r in robots
            ])
            
            # Fit quadratic model
            theta = self.estimator.fit(rel_pos, readings)
            gradient = self.estimator.extract_gradient(theta)
            hessian = self.estimator.extract_hessian(theta)
            
            # Record history
            self.history['centroids'].append(centroid.copy())
            self.history['gradients'].append(gradient.copy())
            self.history['hessians'].append(hessian.copy())
            
            # Compute and apply Newton step
            step = self.stepper.compute_step(gradient, hessian)
            centroid += step
        
        self.final_centroid = centroid.copy()
        return centroid

## Visualization: 4-Panel Diagnostic Plot

**Panel (0,0):** 2D contour plot with trajectory, start marker (blue star), end marker (red star), and true saddle (black X).

**Panel (0,1):** 3D surface with trajectory projected onto it.

**Panel (1,0):** Gradient magnitude vs. iteration (semilogy scale).

**Panel (1,1):** Distance to the true saddle (origin) vs. iteration.

In [113]:
class Visualizer:
    """Diagnostic visualization for saddle point navigation."""
    
    def __init__(self, field, true_saddle=(0.0, 0.0)):
        """
        Args:
            field: ScalarField instance
            true_saddle: (x, y) coordinate of the true saddle point
        """
        self.field = field
        self.true_saddle = np.array(true_saddle)
    
    def plot(self, navigator, title='6-Robot Pentagon: Newton Step (No Rotation)'):
        """Create and display the 4-panel diagnostic figure.
        
        Args:
            navigator: SaddlePointNavigator6 instance (after navigate() has been called)
            title: figure title
        """
        centroids = np.array(navigator.history['centroids'])
        gradients = np.array(navigator.history['gradients'])
        
        # Compute gradient magnitudes and distances to saddle
        grad_norms = np.linalg.norm(gradients, axis=1)
        distances = np.linalg.norm(centroids - self.true_saddle, axis=1)
        
        fig, axes = plt.subplots(2, 2, figsize=(12, 10))
        fig.suptitle(title, fontsize=14, fontweight='bold')
        
        # Panel (0, 0): 2D contour + trajectory
        ax = axes[0, 0]
        cs = self.field.plot_contour(ax, x_range=(-3, 3), y_range=(-3, 3))
        ax.plot(centroids[:, 0], centroids[:, 1], 'r-', linewidth=1.5, alpha=0.8, label='Trajectory')
        ax.plot(navigator.initial_centroid[0], navigator.initial_centroid[1], 'b*', markersize=15, label='Start')
        ax.plot(navigator.final_centroid[0], navigator.final_centroid[1], 'r*', markersize=15, label='End')
        ax.plot(self.true_saddle[0], self.true_saddle[1], 'kx', markersize=10, markeredgewidth=2, label='True Saddle')
        ax.set_xlabel('x')
        ax.set_ylabel('y')
        ax.set_title('2D Trajectory')
        ax.legend(fontsize=9)
        ax.set_aspect('equal')
        
        # Panel (0, 1): 3D surface + trajectory
        ax = axes[0, 1]
        ax.remove()
        ax = fig.add_subplot(2, 2, 2, projection='3d')
        self.field.plot_surface(ax, x_range=(-3, 3), y_range=(-3, 3))
        z_traj = np.array([self.field.evaluate(c[0], c[1]) for c in centroids])
        ax.plot(centroids[:, 0], centroids[:, 1], z_traj, 'r-', linewidth=2, label='Trajectory')
        ax.scatter([navigator.initial_centroid[0]], [navigator.initial_centroid[1]],
                   [self.field.evaluate(*navigator.initial_centroid)], c='b', s=100, marker='*', label='Start')
        ax.scatter([navigator.final_centroid[0]], [navigator.final_centroid[1]],
                   [self.field.evaluate(*navigator.final_centroid)], c='r', s=100, marker='*', label='End')
        ax.set_xlabel('x')
        ax.set_ylabel('y')
        ax.set_zlabel('σ')
        ax.set_title('3D Surface + Trajectory')
        ax.legend(fontsize=9)
        
        # Panel (1, 0): Gradient magnitude vs iteration
        ax = axes[1, 0]
        ax.semilogy(grad_norms, 'g-', linewidth=1.5)
        ax.set_xlabel('Iteration')
        ax.set_ylabel('||∇σ||')
        ax.set_title('Gradient Magnitude (log scale)')
        ax.grid(True, alpha=0.3)
        
        # Panel (1, 1): Distance to saddle vs iteration
        ax = axes[1, 1]
        ax.semilogy(distances, 'b-', linewidth=1.5)
        ax.set_xlabel('Iteration')
        ax.set_ylabel('Distance to Saddle')
        ax.set_title('Distance to True Saddle (log scale)')
        ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        # Print summary
        print(f"\nSummary:")
        print(f"  Initial centroid: {navigator.initial_centroid}")
        print(f"  Final centroid:   {navigator.final_centroid}")
        print(f"  Final distance to saddle: {distances[-1]:.6f}")
        print(f"  Final gradient norm:      {grad_norms[-1]:.6f}")

## Run the Simulation

Configure the simulation parameters below, then run the cell to execute the navigation and display diagnostics.

In [114]:
# ========== Configuration Parameters ==========
# Formation geometry
ROBOT_DISTANCE = 0.2     # radius of pentagon ring
BASE_ANGLE = 0.0          # fixed orientation (no rotation control)

# Newton step
STEP_SIZE = 0.02          # multiplier on Newton delta
MAX_STEP = 1.0            # maximum magnitude of delta before clipping

# Simulation
N_ITERATIONS = 500        # number of Newton steps
START_POSITION = [0.59, -2.5]  # initial centroid

# Random seed (for reproducibility, optional)
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# ========== Instantiate and Run ==========
field = ScalarField()
formation = PentagonFormation(radius=ROBOT_DISTANCE, base_angle=BASE_ANGLE)
estimator = QuadraticEstimator()
#stepper = NewtonStepper(step_size=STEP_SIZE, max_step=MAX_STEP)
stepper = AnisotropicNewtonStepper(gain_perp=0.5, gain_along=0.02)
navigator = SaddlePointNavigator6(
    formation, estimator, stepper, field,
    n_iterations=N_ITERATIONS
)

final_pos = navigator.navigate(START_POSITION)
print(f"Navigation complete. Final position: {final_pos}")

# ========== Visualize ==========
viz = Visualizer(field, true_saddle=(0.0, 0.0))
viz.plot(navigator, title='6-Robot Pentagon Formation: Newton Step (No Rotation)')

TypeError: __init__() got an unexpected keyword argument 'step_size'

## Sign-Decomposed Newton Step

**Alternative step rule:** Decompose the Newton direction into per-axis unit contributions.

The Newton step $\Delta = -H^{-1}\nabla\sigma$ normally combines information from both axes with its natural magnitude. Here, we instead extract the **sign** of each component and step one unit per axis:

$$\mathbf{step} = \text{step\_size} \cdot \text{sign}(\Delta)$$

This equalizes the axis contributions: each axis contributes exactly $\pm\text{step\_size}$, regardless of the Newton direction's aspect ratio. The combined step magnitude is always $\text{step\_size} \cdot \sqrt{2}$ (a pure diagonal). The **direction** (which quadrant) still comes from Newton.

This is an unusual step rule — useful for exploring how axis-aligned constraints affect convergence.

In [ ]:
class SignDecomposedNavigator:
    """Newton direction decomposed into per-axis unit contributions.
    
    The Newton step -H⁻¹∇σ determines the quadrant (sign of dx, dy).
    Each axis then contributes exactly ±step_size independently.
    Combined step magnitude is always step_size * sqrt(2).
    """
    
    def __init__(self, formation, estimator, stepper, field, n_iterations=300):
        self.formation    = formation
        self.estimator    = estimator
        self.stepper      = stepper   # only step_size is used
        self.field        = field
        self.n_iterations = n_iterations
        self.history = {
            'centroids':    [],
            'gradients':    [],
            'hessians':     [],
            'newton_steps': [],   # raw Newton delta (before sign decomposition)
        }

    def _sign_step(self, gradient, hessian):
        """Compute sign-decomposed step from Newton direction."""
        det = np.linalg.det(hessian)
        if np.abs(det) > 1e-10:
            delta = -np.linalg.solve(hessian, gradient)
        else:
            delta = -np.linalg.pinv(hessian) @ gradient
        return self.stepper.step_size * np.sign(delta), delta

    def navigate(self, start_centroid):
        centroid = np.array(start_centroid, dtype=float)
        self.initial_centroid = centroid.copy()

        for _ in range(self.n_iterations):
            robots   = self.formation.robot_positions(centroid)
            rel_pos  = self.formation.relative_positions(centroid)
            readings = np.array([self.field.evaluate(r[0], r[1]) for r in robots])

            theta    = self.estimator.fit(rel_pos, readings)
            gradient = self.estimator.extract_gradient(theta)
            hessian  = self.estimator.extract_hessian(theta)

            step, raw_delta = self._sign_step(gradient, hessian)

            self.history['centroids'].append(centroid.copy())
            self.history['gradients'].append(gradient.copy())
            self.history['hessians'].append(hessian.copy())
            self.history['newton_steps'].append(raw_delta.copy())

            centroid += step

        self.final_centroid = centroid.copy()
        return centroid

In [ ]:
# ========== Sign-Decomposed Configuration ==========
ROBOT_DISTANCE_SD = 0.2
BASE_ANGLE_SD     = 0.0
STEP_SIZE_SD      = 0.02
MAX_STEP_SD       = 1.0
N_ITERATIONS_SD   = 300
START_SD          = [0.5, -2.5]

np.random.seed(42)

field_sd     = ScalarField()
formation_sd = PentagonFormation(radius=ROBOT_DISTANCE_SD, base_angle=BASE_ANGLE_SD)
estimator_sd = QuadraticEstimator()
stepper_sd   = NewtonStepper(step_size=STEP_SIZE_SD, max_step=MAX_STEP_SD)
nav_sd       = SignDecomposedNavigator(
    formation_sd, estimator_sd, stepper_sd, field_sd,
    n_iterations=N_ITERATIONS_SD
)

nav_sd.navigate(START_SD)

centroids     = np.array(nav_sd.history['centroids'])
gradients     = np.array(nav_sd.history['gradients'])
newton_steps  = np.array(nav_sd.history['newton_steps'])

grad_norms = np.linalg.norm(gradients, axis=1)
distances  = np.linalg.norm(centroids - np.array([0.0, 0.0]), axis=1)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle('Sign-Decomposed Newton Step', fontsize=14, fontweight='bold')

# Panel (0,0): 2D contour + trajectory
ax = axes[0, 0]
field_sd.plot_contour(ax, x_range=(-3, 3), y_range=(-3, 3))
ax.plot(centroids[:, 0], centroids[:, 1], 'r-', linewidth=1.5, alpha=0.8, label='Trajectory')
ax.plot(*nav_sd.initial_centroid, 'b*', markersize=15, label='Start')
ax.plot(*nav_sd.final_centroid,   'r*', markersize=15, label='End')
ax.plot(0, 0, 'kx', markersize=10, markeredgewidth=2, label='True Saddle')
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_title('2D Trajectory')
ax.legend(fontsize=9); ax.set_aspect('equal')

# Panel (0,1): 3D surface + trajectory
ax = axes[0, 1]; ax.remove()
ax = fig.add_subplot(2, 2, 2, projection='3d')
field_sd.plot_surface(ax, x_range=(-3, 3), y_range=(-3, 3))
z_traj = np.array([field_sd.evaluate(c[0], c[1]) for c in centroids])
ax.plot(centroids[:, 0], centroids[:, 1], z_traj, 'r-', linewidth=2)
ax.scatter([nav_sd.initial_centroid[0]], [nav_sd.initial_centroid[1]],
           [field_sd.evaluate(*nav_sd.initial_centroid)], c='b', s=100, marker='*')
ax.scatter([nav_sd.final_centroid[0]], [nav_sd.final_centroid[1]],
           [field_sd.evaluate(*nav_sd.final_centroid)], c='r', s=100, marker='*')
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('σ')
ax.set_title('3D Surface + Trajectory')

# Panel (1,0): Gradient magnitude
ax = axes[1, 0]
ax.semilogy(grad_norms, 'g-', linewidth=1.5)
ax.set_xlabel('Iteration'); ax.set_ylabel('||∇σ||')
ax.set_title('Gradient Magnitude (log scale)')
ax.grid(True, alpha=0.3)

# Panel (1,1): Distance to saddle
ax = axes[1, 1]
ax.semilogy(distances, 'b-', linewidth=1.5)
ax.set_xlabel('Iteration'); ax.set_ylabel('Distance to Saddle')
ax.set_title('Distance to True Saddle (log scale)')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nSign-Decomposed Summary:")
print(f"  Final position:           {nav_sd.final_centroid}")
print(f"  Final distance to saddle: {distances[-1]:.6f}")
print(f"  Final gradient norm:      {grad_norms[-1]:.6f}")